# Lab Gouvernance — OpenMetadata + Lineage · Jour 3

**Profil docker :** `governance` · **Durée :** 70 min · **Format :** Trio rôlé

---

## Problème sans gouvernance

```
Analyst : "D'où vient ce chiffre dans mon dashboard ?"
Engineer : "Je sais pas, y'a eu une migration en janvier."
DPO : "Monsieur, vous avez 48h pour supprimer toutes les données de ce client."
Engineer : "Dans quelle table ? Il y en a 47 qui contiennent customer_id..."
```

## Ce que ce lab apporte

- **Catalogage automatique** des tables Iceberg via Polaris
- **Data lineage** : de la source Postgres jusqu'au dashboard
- **Data contracts** sur le catalog Polaris
- **RGPD** : effacement ciblé avec Iceberg CoW/MoR
- **Observation** des formats réels dans MinIO (teaser formats)

---

## Contexte

> Le DPO de RetailCo reçoit une demande d'effacement GDPR pour le client `cust-8821`.
> Il a 48h. Votre mission : trouver toutes les tables qui contiennent ce client,
> documenter la dépendance, effectuer l'effacement et prouver la conformité.

In [1]:
import os, json, boto3, time, requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── JARs Iceberg / Nessie / S3A (volume spark-jars monté dans /opt/spark-jars)
JARS = (
    "/opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.11.0.jar,"
    "/opt/spark-jars/iceberg-aws-bundle-1.11.0.jar,"
    "/opt/spark-jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar,"
    "/opt/spark-jars/hadoop-aws-3.3.4.jar,"
    "/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
)

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("RetailCo-Governance")
    .config("spark.jars", JARS)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    # Catalog Polaris (REST)
    .config("spark.sql.catalog.polaris",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.polaris.uri",          os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog"))
    .config("spark.sql.catalog.polaris.warehouse",    "retailco")         # nom du catalog Polaris
    .config("spark.sql.catalog.polaris.credential",   os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t"))
    .config("spark.sql.catalog.polaris.scope",        "PRINCIPAL_ROLE:ALL")
    # S3FileIO config MinIO — requis par ResolvingFileIO (AWS SDK v2)
    .config("spark.sql.catalog.polaris.s3.endpoint",          os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.polaris.s3.access-key-id",     os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.secret-access-key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.path-style-access",  "true")
    .config("spark.sql.catalog.nessie.s3.endpoint",           os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.nessie.s3.access-key-id",      os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.secret-access-key",  os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.path-style-access",   "true")
    # HadoopFileIO — évite ResolvingFileIO qui requiert AWS SDK v2
    # Source : iceberg.apache.org/docs/latest/aws/#s3-file-io
    # Catalog Nessie
    .config("spark.sql.catalog.nessie",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          os.getenv("NESSIE_URI_V1", "http://nessie:19120/api/v1"))
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    "s3a://warehouse/nessie/")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",             os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key",           os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key",           os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.impl",                 "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.defaultCatalog", "polaris")
    .getOrCreate()
)

print(f"✅ Spark {spark.version}")
print(f"✅ Catalog par défaut : polaris")
print(f"✅ Iceberg JARs chargés depuis /opt/spark-jars/")

# ── Clients ───────────────────────────────────────────────────────────────────
OM_API  = "http://openmetadata:8585/api/v1"
OM_AUTH = ("admin", "admin")
MINIO   = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
POLARIS = os.getenv("POLARIS_URI",    "http://polaris:8181/api/catalog")

s3 = boto3.client("s3", endpoint_url=MINIO,
    aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")

def om(method, path, data=None):
    r = requests.request(method, f"{OM_API}{path}",
        auth=OM_AUTH,
        headers={"Content-Type": "application/json"},
        json=data)
    try:
        return r.json()
    except Exception:
        return {"status_code": r.status_code}
print("✅ Clients OpenMetadata et MinIO prêts")

✅ Spark 3.5.0
✅ Catalog par défaut : polaris
✅ Iceberg JARs chargés depuis /opt/spark-jars/
✅ Clients OpenMetadata et MinIO prêts


---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Observer les formats réels dans MinIO — Parquet data files, JSON metadata, manifests.

In [ ]:
# ── Teaser formats : observer les fichiers réels dans MinIO ────────────────────
print('📁 Exploration du stockage MinIO — formats réels :')
print('='*60)

buckets_to_scan = ['warehouse', 'bronze', 'silver', 'gold']
for bucket in buckets_to_scan:
    try:
        resp = s3.list_objects_v2(Bucket=bucket)
        files = resp.get('Contents', [])
        if files:
            print(f'\ns3://{bucket}/ ({len(files)} fichiers) :')
            by_ext = {}
            for obj in files:
                ext = obj['Key'].split('.')[-1] if '.' in obj['Key'] else 'dir'
                by_ext.setdefault(ext, []).append(obj)
            for ext, objs in sorted(by_ext.items()):
                total_size = sum(o['Size'] for o in objs)
                sample = objs[0]['Key'].split('/')[-1]
                print(f'   .{ext:<8} × {len(objs):>3}  ({total_size/1024:.0f} Ko)  ex: {sample}')
    except Exception as e:
        pass

print('\n💡 Ce que vous voyez :')
print('   .parquet → fichiers de données (compressés, colonnaires)')
print('   .json    → metadata Iceberg (manifest list, snapshot)')
print('   .avro    → commit log Iceberg (ou format streaming)')
print('   .metadata.json → schéma + historique des snapshots')

In [ ]:
# ── Inspecter un fichier Parquet directement ─────────────────────────────────
try:
    import pyarrow.parquet as pq
    import io

    resp = s3.list_objects_v2(Bucket='warehouse', Prefix='retailco/')
    parquet_files = [obj['Key'] for obj in resp.get('Contents', [])
                     if obj['Key'].endswith('.parquet')]

    if parquet_files:
        key = parquet_files[0]
        data = s3.get_object(Bucket='warehouse', Key=key)['Body'].read()
        pf = pq.read_table(io.BytesIO(data))
        print(f'📄 Fichier Parquet : {key.split("/")[-1]}')
        print(f'   Lignes     : {pf.num_rows}')
        print(f'   Colonnes   : {pf.num_columns}')
        print(f'   Schéma     : {[(f.name, str(f.type)) for f in pf.schema][:5]}')
        print(f'   Taille     : {len(data)/1024:.1f} Ko')
        print(f'   Compression: Snappy (défaut Iceberg)')
    else:
        print('⚠️  Pas encore de fichiers Parquet — créer des tables Iceberg d\'abord')
except ImportError:
    subprocess.run(['pip', 'install', 'pyarrow', '--quiet'], check=False)
    print('PyArrow installé — relancer la cellule')

---
## ⏱️ Temps 2 — Incident à résoudre (45 min)

### Partie A — RGPD : effacement ciblé (20 min)

> **Mission DPO** : effacer toutes les données du client `cust-8821` dans les 48h.
> Vous devez d'abord trouver toutes les tables qui contiennent ce customer_id,
> puis effectuer l'effacement et prouver qu'il est complet.

**TODO 1** : Trouver toutes les tables Iceberg contenant `cust-8821`, les effacer, vérifier.

In [ ]:
# ── Préparer les tables (simuler une situation avec données client répandues) ──
TARGET_CUSTOMER = 'cust-8821'

# Créer des tables avec ce client dans plusieurs couches
for layer, table in [('polaris.retailco', 'transactions_silver'),
                     ('polaris.retailco', 'customer_profiles'),
                     ('polaris.mlops', 'churn_scores')]:
    cat_name = layer.split('.')[0]
    ns_name  = layer.split('.')[1]
    spark.sql(f'USE {cat_name}')
    spark.sql(f'CREATE NAMESPACE IF NOT EXISTS {ns_name}')

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS polaris.retailco.transactions_silver
    (transaction_id STRING, customer_id STRING, store_country STRING,
     total_amount_xof BIGINT, transaction_ts TIMESTAMP)
    USING iceberg TBLPROPERTIES ('write.delete.mode'='copy-on-write')
""")
spark.sql(f"""
    INSERT INTO polaris.retailco.transactions_silver VALUES
    ('txn-001', '{TARGET_CUSTOMER}', 'Cote d Ivoire', 395000, current_timestamp()),
    ('txn-002', 'cust-0001', 'Sénégal', 78700, current_timestamp()),
    ('txn-003', '{TARGET_CUSTOMER}', 'Sénégal', 32000, current_timestamp())
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.retailco.customer_profiles
    (customer_id STRING, email STRING, nom STRING, prenom STRING, created_at TIMESTAMP)
    USING iceberg TBLPROPERTIES ('write.delete.mode'='copy-on-write')
""")
spark.sql(f"""
    INSERT INTO polaris.retailco.customer_profiles VALUES
    ('{TARGET_CUSTOMER}', 'client8821@email.com', 'Martin', 'Jean', current_timestamp()),
    ('cust-0001', 'autre@email.com', 'Dupont', 'Marie', current_timestamp())
""")

print(f'✅ Tables prêtes — client {TARGET_CUSTOMER} présent dans 2 tables')

In [ ]:
# TODO 1 — RGPD : effacement complet
# 📝 a) Scanner toutes les tables pour trouver customer_id = TARGET_CUSTOMER
#    b) Effectuer un DELETE ACID sur chaque table
#    c) Vérifier que la donnée a disparu
#    d) Vérifier que l'historique Iceberg (snapshots) ne permet plus d'y accéder
#       après un VACUUM (ou noter que le time travel doit aussi être géré)

print(f'💡 Tables à scanner : polaris.retailco.transactions_silver, polaris.retailco.customer_profiles')
print(f'💡 spark.sql("DELETE FROM ... WHERE customer_id = \'{TARGET_CUSTOMER}\'")')
print('💡 spark.sql("SELECT count(*) FROM ... WHERE customer_id = ...") pour vérifier')
print('💡 spark.sql("SELECT * FROM table.snapshots") pour voir l\'historique')

In [ ]:
# ✅ SOLUTION TODO 1

TABLES_PII = [
    'polaris.retailco.transactions_silver',
    'polaris.retailco.customer_profiles',
]

print(f'🔍 Recherche du client {TARGET_CUSTOMER} dans toutes les tables PII...')
print()

delete_report = []
for table in TABLES_PII:
    # Scanner
    count_before = spark.sql(f"SELECT count(*) AS n FROM {table} WHERE customer_id = '{TARGET_CUSTOMER}'").collect()[0]['n']
    print(f'   {table}')
    print(f'   → Lignes trouvées : {count_before}')

    if count_before > 0:
        # DELETE ACID Iceberg
        spark.sql(f"DELETE FROM {table} WHERE customer_id = '{TARGET_CUSTOMER}'")
        count_after = spark.sql(f"SELECT count(*) AS n FROM {table} WHERE customer_id = '{TARGET_CUSTOMER}'").collect()[0]['n']
        print(f'   → Après DELETE   : {count_after} lignes (✅ effacé)' if count_after == 0 else f'   ⚠️  {count_after} lignes restantes')

        # Snapshots Iceberg restants
        snapshots = spark.sql(f'SELECT count(*) AS n FROM {table}.snapshots').collect()[0]['n']
        print(f'   → Snapshots Iceberg : {snapshots} (time travel possible — VACUUM requis pour conformité stricte)')
        delete_report.append({'table': table, 'rows_deleted': count_before, 'status': 'deleted'})
    else:
        print(f'   → Client absent')
    print()

print('📋 Rapport RGPD :')
for r in delete_report:
    print(f'   ✅ {r["table"]} : {r["rows_deleted"]} ligne(s) supprimée(s)')
print(f'\n⚠️  Note conformité : les snapshots Iceberg historiques contiennent encore les données.')
print('   Solution : VACUUM avec retention = 0h (à faire hors heures de production)')
print('   spark.conf.set("spark.sql.catalog.polaris.io-impl", ...) + CALL ... rewrite_manifests')

### Partie B — Data Lineage via OpenMetadata (15 min)

**TODO 2** : Enregistrez une connexion de lineage dans OpenMetadata pour documenter
que `transactions_silver` est dérivée de `transactions_bronze`.

In [ ]:
# TODO 2 — Lineage dans OpenMetadata
# 📝 a) Créer ou trouver les tables dans OpenMetadata
#    b) Créer une relation de lineage Bronze → Silver → Gold
#    c) Requêter le lineage pour vérifier

print('💡 om("GET", "/tables?limit=10") pour voir les tables connues')
print('💡 om("PUT", "/lineage", data={"edge": {"fromEntity": ..., "toEntity": ...}})')
print('💡 om("GET", "/lineage/table/<id>?upstreamDepth=2") pour voir le lineage')

In [ ]:
# ✅ SOLUTION TODO 2

# Voir les tables déjà ingérées
tables_resp = om('GET', '/tables?limit=10')
print('📋 Tables dans OpenMetadata :')
tables = tables_resp.get('data', [])
for t in tables[:5]:
    print(f"   {t.get('fullyQualifiedName', '')}  (id: {t.get('id', '')[:8]}...)")

# Créer des entrées de tables manuellement si OpenMetadata n'a pas encore ingéré
# En production : utiliser le connecteur Polaris → OpenMetadata
lineage_data = {
    'edge': {
        'fromEntity': {
            'type': 'table',
            'fullyQualifiedName': 'polaris.retailco.transactions_bronze'
        },
        'toEntity': {
            'type': 'table',
            'fullyQualifiedName': 'polaris.retailco.transactions_silver'
        },
        'lineageDetails': {
            'transformationType': 'VIEW',
            'description': 'Pipeline dbt silver_transactions — nettoyage + déduplication'
        }
    }
}

resp = om('PUT', '/lineage', data=lineage_data)
print(f'\n📊 Lineage créé : {resp.get("type", "OK")}')
print('\n✅ Le lineage Bronze → Silver est maintenant documenté dans OpenMetadata.')
print('   En production : le connecteur Iceberg/dbt remplit ce lineage automatiquement.')
print('   Un analyste peut maintenant retracer la source de n\'importe quel chiffre.')

print('\n🔗 Voir dans OpenMetadata UI : http://localhost:8585')
print('   Explore → Tables → retailco → transactions_silver → Lineage')

### Partie C — Data Contracts sur Polaris (10 min)

In [ ]:
# ── Data Contract : propriétés de namespace comme contrat ────────────────────
print('📋 Data Contract — namespace retailco :')

# Mettre à jour les propriétés du namespace avec les engagements
contract = {
    'updates': {
        'owner':           'analytics-team@retailco.com',
        'sla_freshness':   '< 2h',
        'sla_quality':     '> 99.5%',
        'schema_version':  '1.0',
        'pii_fields':      'customer_id,email,nom,prenom',
        'rgpd_compliant':  'true',
        'retention_days':  '1095',  # 3 ans
        'on_call':         'data-engineering-team',
    },
    'removals': []
}

# Le chemin REST Iceberg/Polaris exige le préfixe du catalog ('retailco')
# avant /namespaces — sans lui, Polaris retourne 404 (namespace introuvable)
resp = requests.post(
    f'{POLARIS}/v1/retailco/namespaces/retailco/properties',
    json=contract,
    headers={'Content-Type': 'application/json'}
)
print(f'   Status : {resp.status_code}')

# Lire les propriétés — GET sur le namespace lui-même (pas /properties, réservé au POST)
props = requests.get(
    f'{POLARIS}/v1/retailco/namespaces/retailco'
).json()
print('\n📄 Propriétés du namespace (= data contract) :')
for k, v in props.get('properties', {}).items():
    print(f'   {k:<20} : {v}')

---
## ⏱️ Temps 3 — Extension (10 min)

Automatiser l'audit RGPD : vérifier que le DELETE a bien été propagé dans tous les snapshots Iceberg actifs.

In [ ]:
# ── Audit RGPD automatisé ────────────────────────────────────────────────────
print(f'🔍 Audit RGPD automatisé pour le client {TARGET_CUSTOMER}')
print('='*55)

audit_results = []
for table in TABLES_PII:
    # Vérifier snapshot actuel
    count_current = spark.sql(f"""
        SELECT count(*) AS n FROM {table}
        WHERE customer_id = '{TARGET_CUSTOMER}'
    """).collect()[0]['n']

    # Vérifier snapshots historiques (time travel)
    snapshots = spark.sql(f'SELECT snapshot_id, committed_at FROM {table}.snapshots ORDER BY committed_at DESC').collect()
    historical_found = False
    for snap in snapshots[1:]:  # Sauter le snapshot courant
        try:
            count_hist = spark.sql(f"""
                SELECT count(*) AS n FROM {table}
                VERSION AS OF {snap['snapshot_id']}
                WHERE customer_id = '{TARGET_CUSTOMER}'
            """).collect()[0]['n']
            if count_hist > 0:
                historical_found = True
                break
        except Exception:
            # Snapshot potentiellement expiré ou inaccessible — on continue le scan
            pass

    status = 'CONFORME' if count_current == 0 else 'NON CONFORME'
    warning = '⚠️  TIME TRAVEL' if historical_found else ''
    audit_results.append({'table': table, 'current': count_current, 'status': status, 'time_travel': historical_found})
    print(f'\n   {table}')
    print(f'   Données actuelles    : {count_current} ligne(s) — {status}')
    print(f'   Accessible time travel: {historical_found} {warning}')

print('\n📋 Résumé audit RGPD :')
for r in audit_results:
    print(f'   {r["status"]} | {r["table"].split(".")[-1]}')
    if r['time_travel']:
        print('   ⚠️  ACTION REQUISE : CALL polaris.system.expire_snapshots() + VACUUM')
print('\n✅ Rapport d\'audit généré — à transmettre au DPO dans les 48h.')

---
## ⏱️ Débrief client (10 min)

### Questions — rôle Client :

1. **"Vous m'avez dit que les données sont effacées. Mais le time travel Iceberg les montre encore. Comment vous gérez ça vis-à-vis de la CNIL ?"**
2. **"Le lineage que vous avez créé dans OpenMetadata — il est maintenu automatiquement ou il faut le mettre à jour à la main ?"**
3. **"Le data contract est dans les propriétés du namespace Polaris. C'est du texte. Comment vous garantissez que les équipes le respectent ?"**
4. **"Si j'ai 200 tables, l'audit RGPD prend combien de temps ? Est-ce que ça scale ?"**

---

## Récapitulatif

| Concept | Exercice | Impact |
|---------|---------|--------|
| Formats observés | Parquet + JSON metadata dans MinIO | Comprendre ce qu'Iceberg écrit vraiment |
| Effacement RGPD CoW | DELETE ACID sur tables Iceberg | Conformité en < 48h |
| Time travel RGPD | Identifier les snapshots exposés | Compléter l'effacement avec expire_snapshots |
| Lineage OpenMetadata | Bronze → Silver documenté | Tracer n'importe quel chiffre jusqu'à la source |
| Data contracts Polaris | Propriétés namespace | Gouvernance décentralisée, audit trail |